# Conversion Prediction Model

# Objective
**Problem:** Predict which free-tier (non-customer) companies are likely to convert to paying customers_df_df_df_df_df_df_df_df_df within the next 30 days.

**Output:** Weekly prioritized list (generated on Sundays).

# Assumptions
+ Alexa rank Null means that the company has no webpage, thus we assign 16000001 to these cases

# Table of Contents

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, IFrame
from ydata_profiling import ProfileReport
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# Custom libs
from data_prep import missing_summary, encode_industries

# Constants

In [2]:
ACTION_COLS = ["ACTIONS_CRM_CONTACTS", "ACTIONS_CRM_COMPANIES", "ACTIONS_CRM_DEALS", "ACTIONS_EMAIL"]
USER_COLS   = ["USERS_CRM_CONTACTS", "USERS_CRM_COMPANIES", "USERS_CRM_DEALS", "USERS_EMAIL"]
METRIC_COLS = ACTION_COLS + USER_COLS
WINDOWS     = {"7d": 7, "14d": 14, "30d": 30, "60d": 60}
PREDICTION_HORIZON_DAYS = 30

# Load Data
- Load data
- Preprocess and standardize columns
- Check data integrity and basic statistics

In [6]:

# Load data
customers_df = pd.read_csv("../data/customers.csv")
noncustomers_df = pd.read_csv("../data/noncustomers.csv")
usage_actions_df = pd.read_csv("../data/usage_actions.csv")

# Parse dates
customers_df['CLOSEDATE'] = pd.to_datetime(customers_df['CLOSEDATE'])
usage_actions_df['WHEN_TIMESTAMP'] = pd.to_datetime(usage_actions_df['WHEN_TIMESTAMP'])

# Sort by CLOSEDATE and WHEN_TIMESTAMP
customers_df = customers_df.sort_values(by=['CLOSEDATE'])
usage_actions_df = usage_actions_df.sort_values(by=['WHEN_TIMESTAMP'])

# All columns to upper case
customers_df.columns = customers_df.columns.str.upper()
noncustomers_df.columns = noncustomers_df.columns.str.upper()
usage_actions_df.columns = usage_actions_df.columns.str.upper()


# Check data
print(f"customers:     {customers_df.shape}  — IDs {customers_df['ID'].min()}–{customers_df['ID'].max()}")
print(f"noncustomers:  {noncustomers_df.shape}  — IDs {noncustomers_df['ID'].min()}–{noncustomers_df['ID'].max()}")
print(f"usage_actions: {usage_actions_df.shape}")
print(f"Usage date range: {usage_actions_df['WHEN_TIMESTAMP'].min().date()} → {usage_actions_df['WHEN_TIMESTAMP'].max().date()}")
print(f"Unique ids: {usage_actions_df['ID'].nunique()}")

cust_ids   = set(customers_df["ID"])
noncust_ids= set(noncustomers_df["ID"])
print(f"ID overlap (customers ∩ noncustomers): {len(cust_ids & noncust_ids)}")  

REF_DATE = usage_actions_df["WHEN_TIMESTAMP"].max()
print(f"Reference date ('today'): {REF_DATE.date()}")


customers:     (200, 6)  — IDs 1–200
noncustomers:  (5003, 4)  — IDs 201–5200
usage_actions: (25387, 10)
Usage date range: 2019-01-07 → 2020-07-27
Unique ids: 3569
ID overlap (customers ∩ noncustomers): 0
Reference date ('today'): 2020-07-27


# EDA
+ Missing values
+ Using ydata_profiling for detailed and shareable insights found in ./reports

In [7]:
# Check missing data
missing_summary(customers_df, "CUSTOMERS")
missing_summary(noncustomers_df, "NON-CUSTOMERS")
missing_summary(usage_actions_df, "USAGE")



CUSTOMERS columns with missing values:
        column  missing_count  missing_pct
      INDUSTRY            129         64.5
EMPLOYEE_RANGE              2          1.0

NON-CUSTOMERS columns with missing values:
        column  missing_count  missing_pct
      INDUSTRY           3725        74.46
EMPLOYEE_RANGE            532        10.63
    ALEXA_RANK            114         2.28

USAGE columns with missing values:
None


In [8]:
# Usage-level comparison: customers vs non-customers
usage_customers_df = usage_actions_df[usage_actions_df["ID"].isin(cust_ids)].copy()
usage_noncustomers_df = usage_actions_df[usage_actions_df["ID"].isin(noncust_ids)].copy()

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

usage_customer_profile = ProfileReport(usage_customers_df, title="Customer Usage", minimal=True)
usage_noncustomer_profile = ProfileReport(usage_noncustomers_df, title="Non-customer Usage", minimal=True)

usage_customer_profile.compare(usage_noncustomer_profile).to_file(reports_dir / "usage_comparison.html")

display(IFrame(src=str(reports_dir / "usage_comparison.html"), width="100%", height=600))

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 6525.05it/s]


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 719.39it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
# Company-level comparison: customers vs non-customers
historical_customers_df = customers_df[customers_df["CLOSEDATE"] <= REF_DATE].copy()

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

customer_profile    = ProfileReport(historical_customers_df, title="Customers",     minimal=True)
noncustomer_profile = ProfileReport(noncustomers_df,         title="Non-customers", minimal=True)

customer_profile.compare(noncustomer_profile).to_file(reports_dir / "company_comparison.html")

display(IFrame(src=str(reports_dir / "company_comparison.html"), width="100%", height=600))

c:\Users\totic\anaconda3\Lib\site-packages\ydata_profiling\compare_reports.py:191: UserWarning: The datasets being profiled have a different set of columns. Only the left side profile will be calculated.
  warnings.warn(


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 954.37it/s]


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 241.01it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# Feature Engineering
+ Assumption: Alexa rank Null means that the company has no webpage, thus we assign 16000001 to these cases
+ INDUSTRY nulls could be reduced by doing Web Scraping or buying a company industry database
+ Employee range to the middle of the range
+ One Hot Encoding of INDUSTRY
+ Build date features

In [10]:
# 1. Add Label and Align Columns
# We use assign to create the label without modifying the original dfs immediately
customers_base = customers_df.assign(CUSTOMER=1)
noncustomers_base = noncustomers_df.assign(CUSTOMER=0)

# 2. Canonical Schema Alignment
# Ensure non-customers have all columns from customers (filled with NaN where missing)
target_cols = customers_base.columns
noncustomers_base = noncustomers_base.reindex(columns=target_cols)

# 3. Bulk Type Alignment
# Map the dtypes from customers directly onto non-customers in one go.
# We convert both to 'nullable' types first to prevent NaN-casting issues.
companies_df = pd.concat([customers_base, noncustomers_base], axis=0, ignore_index=True)

# Force specific critical types (like Datetime) if convert_dtypes is too conservative
companies_df['CLOSEDATE'] = pd.to_datetime(companies_df['CLOSEDATE'], errors='coerce')

print(f"Merged companies_df shape: {companies_df.shape}")
print(companies_df["CUSTOMER"].value_counts(dropna=False))

Merged companies_df shape: (5203, 7)
CUSTOMER
0    5003
1     200
Name: count, dtype: int64


C:\Users\totic\AppData\Local\Temp\ipykernel_31232\3376098774.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  companies_df = pd.concat([customers_base, noncustomers_base], axis=0, ignore_index=True)


In [11]:
# For null ALEXA_RANK assume they where not found and page-rank will be weak
max_rank = companies_df['ALEXA_RANK'].max()
companies_df['ALEXA_RANK'] = np.where(companies_df['ALEXA_RANK'].isnull(), max_rank, companies_df['ALEXA_RANK'])

In [12]:
# Employee range to numberic
# For a better comparison, map EMPLOYEE_RANGE with explicit dictionaries.
EMPLOYEE_RANGE_TO_MID = {
    "1": 1.0,
    "2 to 5": 3.0,
    "6 to 10": 8.0,
    "11 to 25": 18.0,
    "26 to 50": 38.0,
    "51 to 200": 125.0,
    "201 to 1000": 600.0,
    "1001 to 10000": 5500.0,
    "10,001 or more": 10001.0,
}

companies_df["EMPLOYEE_RANGE"] = companies_df["EMPLOYEE_RANGE"].map(EMPLOYEE_RANGE_TO_MID)


In [13]:
# Encoding Insustry
companies_df = encode_industries(companies_df, 'INDUSTRY', max_cats=10)
companies_df

,CLOSEDATE,MRR,ALEXA_RANK,EMPLOYEE_RANGE,INDUSTRY,ID,CUSTOMER,IND_CONSULTING,IND_EDUCATION,IND_FINANCE,IND_SERVICES,IND_MARKETING,IND_OTHER,IND_REAL ESTATE,IND_TECH,IND_UNKNOWN,IND_sklearn
0,2019-01-15,300.96,16000001.0,8.0,NaN,42,1,0,0,0,0,0,0,0,0,1,0
1,2019-01-17,49.07,10390859.0,3.0,NaN,20,1,0,0,0,0,0,0,0,0,1,0
2,2019-01-27,209.98,273063.0,38.0,Technology - Software,174,1,0,0,0,0,0,0,0,1,0,0
3,2019-01-30,40.00,2610402.0,3.0,RENEWABLES_ENVIRONMENT,1,1,0,0,0,0,0,0,0,0,0,1
4,2019-01-30,400.00,16000001.0,18.0,NaN,25,1,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5198,NaT,NaN,16000001.0,NaN,NaN,637,0,0,0,0,0,0,0,0,0,1,0
5199,NaT,NaN,20183.0,5500.0,Non-Profit/Educational Institution,4921,0,0,0,0,1,0,0,0,0,0,0
5200,NaT,NaN,16000001.0,8.0,NaN,1215,0,0,0,0,0,0,0,0,0,1,0
5201,NaT,NaN,16000001.0,18.0,NaN,2693,0,0,0,0,0,0,0,0,0,1,0


In [14]:
usage_actions_df

,WHEN_TIMESTAMP,ACTIONS_CRM_CONTACTS,ACTIONS_CRM_COMPANIES,ACTIONS_CRM_DEALS,ACTIONS_EMAIL,USERS_CRM_CONTACTS,USERS_CRM_COMPANIES,USERS_CRM_DEALS,USERS_EMAIL,ID
2287,2019-01-07,0,0,0,0,0,0,0,0,4916
2346,2019-01-07,0,0,0,0,0,0,0,0,4432
12713,2019-01-07,136,0,1,0,11,0,1,0,157
12827,2019-01-07,1,1,19,0,1,1,1,0,162
13124,2019-01-07,1,0,0,0,1,0,0,0,4412
...,...,...,...,...,...,...,...,...,...,...
7000,2020-07-27,121,343,94,0,9,14,11,0,51
12384,2020-07-27,1,0,0,0,1,0,0,0,3041
20639,2020-07-27,166,0,106,1,7,0,6,1,32
12307,2020-07-27,22,0,3,0,2,0,1,0,4834


In [20]:
def _ols_slope(series: pd.Series) -> float:
    """OLS slope of series over integer positions. Used for trend features."""
    y = series.dropna().values
    if len(y) < 2:
        return 0.0
    x = np.arange(len(y), dtype=float) - np.arange(len(y), dtype=float).mean()
    denom = (x * x).sum()
    return float((x * y).sum() / denom) if denom else 0.0


# ============================================================
# BACKTESTER
# ============================================================

class VectorizedUsageFeatureBacktester:
    """
    Leakage-safe, vectorized feature panel for time-based backtesting.

    Produces a MultiIndex (ID, cutoff) DataFrame where each row is the
    feature state of a portal using ONLY data strictly before the cutoff.

    Cutoffs: last usage date + n_cutoffs Mondays spaced 1 month apart.

    Leakage guards:
      - Rolling features computed on pre-cutoff slice only
      - Recency dates derived from pre-cutoff slice (no negative days)
      - Entropy/diversity derived from pre-cutoff slice
    """

    def __init__(
        self,
        usage_df: pd.DataFrame,
        companies_df: pd.DataFrame,
        n_cutoffs: int = 6,
        action_cols: list = ACTION_COLS,
        user_cols: list = USER_COLS,
        windows: dict = WINDOWS,
    ):
        self.portal_ids  = companies_df["ID"].unique()
        self.n_cutoffs   = n_cutoffs
        self.action_cols = action_cols
        self.user_cols   = user_cols
        self.windows     = windows

        self.df      = self._prepare(usage_df)
        self.cutoffs = self._generate_cutoffs()

    # ── data prep ────────────────────────────────────────────────

    def _prepare(self, usage_df: pd.DataFrame) -> pd.DataFrame:
        df = usage_df.copy()
        df["WHEN_TIMESTAMP"] = pd.to_datetime(df["WHEN_TIMESTAMP"])
        df = df[df["ID"].isin(self.portal_ids)].sort_values(["ID", "WHEN_TIMESTAMP"])
        df["TOTAL_ACTIONS"] = df[self.action_cols].sum(axis=1)
        df["TOTAL_USERS"]   = df[self.user_cols].sum(axis=1)
        df["ACTIVE_DAY"]    = (df["TOTAL_ACTIONS"] > 0).astype(int)
        return df

    # ── cutoffs ───────────────────────────────────────────────────

    def _generate_cutoffs(self) -> list:
        """Last usage date + n_cutoffs Mondays, each 1 month apart. Deduped."""
        last = self.df["WHEN_TIMESTAMP"].max()
        snap = lambda dt: dt - pd.Timedelta(days=dt.weekday())   # nearest Monday ≤ dt
        anchor = snap(last)
        mondays = [snap(anchor - pd.DateOffset(months=i)) for i in range(self.n_cutoffs)]
        return list(dict.fromkeys([last] + mondays))              # last_date first, deduped

    # ── rolling features ─────────────────────────────────────────

    def _rolling_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """All rolling-window features on a pre-filtered slice."""
        grouped = df.set_index("WHEN_TIMESTAMP").groupby("ID")
        frames  = []

        for name, w in self.windows.items():
            roll = grouped.rolling(f"{w}D")

            f = pd.DataFrame({
                # Volume: sum is the only meaningful aggregate —
                # mean/std/max degenerate to the same value for portals
                # with 1 active day in the window (the common case here).
                f"actions_sum_{name}":  roll["TOTAL_ACTIONS"].sum(),
                f"users_sum_{name}":    roll["TOTAL_USERS"].sum(),
                # Frequency: how many days had any activity in the window
                f"active_days_{name}":  roll["ACTIVE_DAY"].sum(),
            })

            # active_ratio: fraction of window days that were active
            f[f"active_ratio_{name}"] = f[f"active_days_{name}"] / w

            # engagement depth: actions per active day (intensity proxy)
            f[f"actions_per_active_day_{name}"] = (
                f[f"actions_sum_{name}"] / (f[f"active_days_{name}"] + 1)
            )

            # trend: OLS slope of daily actions — are they ramping up or down?
            f[f"actions_trend_{name}"] = (
                grouped["TOTAL_ACTIONS"].rolling(f"{w}D").apply(_ols_slope, raw=False)
            )

            # per-module action sums (keep sums only; pct already captures the mix)
            for col in self.action_cols:
                f[f"{col.lower()}_sum_{name}"] = roll[col].sum()

            # user sums per module (volume signal per object type)
            for col in self.user_cols:
                f[f"{col.lower()}_sum_{name}"] = roll[col].sum()

            # module share: what fraction of actions went to each module?
            # Replaces per-module means (redundant with sums at 1 obs/window)
            total = f[f"actions_sum_{name}"].replace(0, np.nan)
            for col in self.action_cols:
                f[f"pct_{col.lower()}_{name}"] = (
                    f[f"{col.lower()}_sum_{name}"] / total
                ).fillna(0)

            frames.append(f)

        return pd.concat(frames, axis=1).reset_index()

    # ── recency features ──────────────────────────────────────────

    def _recency_features(self, df: pd.DataFrame, cutoff: pd.Timestamp) -> pd.DataFrame:
        """Last/first usage dates and derived recency columns. Computed on pre-cutoff slice."""
        active = df[df["TOTAL_ACTIONS"] > 0].groupby("ID")["WHEN_TIMESTAMP"]
        rec = pd.DataFrame({
            "last_usage_date":  active.max(),
            "first_usage_date": active.min(),
        }).reset_index()

        rec["days_since_last_usage"]  = (cutoff - rec["last_usage_date"]).dt.days
        rec["days_since_first_usage"] = (cutoff - rec["first_usage_date"]).dt.days
        rec["usage_tenure_days"]      = (rec["last_usage_date"] - rec["first_usage_date"]).dt.days
        rec["recency_score"]          = 1 / (rec["days_since_last_usage"] + 1)
        return rec.drop(columns=["last_usage_date", "first_usage_date"])

    # ── entropy features ──────────────────────────────────────────

    def _entropy_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Module entropy and diversity. Computed on pre-cutoff slice."""
        totals = df.groupby("ID")[self.action_cols].sum()
        probs  = totals.div(totals.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
        return pd.DataFrame({
            "ID":               totals.index,
            "module_entropy":   (-(probs * np.log(probs + 1e-9)).sum(axis=1)).clip(0).values,
            "module_diversity": (totals > 0).sum(axis=1).values,
        })

    # ── build ─────────────────────────────────────────────────────

    def build(self) -> pd.DataFrame:
        """Build and return the full (ID × cutoff) feature panel."""
        snapshots = []

        for cutoff in self.cutoffs:
            sl = self.df[self.df["WHEN_TIMESTAMP"] < cutoff]
            if sl.empty:
                continue

            snap = (
                self._rolling_features(sl)
                .sort_values(["ID", "WHEN_TIMESTAMP"])
                .groupby("ID").tail(1)
            ).copy()
            snap["cutoff"] = cutoff

            snap = snap.merge(self._recency_features(sl, cutoff), on="ID", how="left")
            snap = snap.merge(self._entropy_features(sl),          on="ID", how="left")
            snapshots.append(snap)

        panel = pd.concat(snapshots, ignore_index=True)

        if {"actions_sum_30d", "actions_sum_60d"}.issubset(panel.columns):
            panel["actions_accel"] = panel["actions_sum_30d"] / (panel["actions_sum_60d"] + 1)

        # Full (ID × cutoff) index — zero-usage portals filled with sentinels
        full_idx = pd.MultiIndex.from_product(
            [self.portal_ids, self.cutoffs], names=["ID", "cutoff"]
        )
        panel = panel.set_index(["ID", "cutoff"]).reindex(full_idx).sort_index()

        # Recency sentinel for never-active portals: large number, not 0
        span = int((self.cutoffs[0] - self.cutoffs[-1]).days + 1)
        for col in ["days_since_last_usage", "days_since_first_usage"]:
            if col in panel.columns:
                panel[col] = panel[col].fillna(span)

        return panel.fillna(0)




In [21]:
# ============================================================
# USAGE
# ============================================================

backtester = VectorizedUsageFeatureBacktester(usage_actions_df, companies_df, n_cutoffs=3)
features   = backtester.build().reset_index()

# Modeling